# LifeMate — ASR public_test (gipformer-68M)

Chép lời mọi audio của public_test (lượt cuối `<id>.wav` + lượt user trong history `<id>_uN.wav`) bằng
`g-group-ai-lab/gipformer-68M-rnnt` (ONNX, sherpa-onnx, chạy CPU). Agent gọi tool chạy ngoài notebook này.

Output trong `/kaggle/working/`:
- `asr_public_test.jsonl`: mỗi dòng một mẫu `{"id", "audio_text", "history_audio_text": [...]}` (history theo thứ tự lượt user).
- `asr_public_test_files.jsonl`: mỗi dòng một file audio `{"audio", "text", "dur_s"}`.
- `asr_train_check.json`: WER trên một phần train để kiểm nhanh chất lượng.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "sherpa-onnx", "soundfile"], check=True)

In [ ]:
import glob, json, os, re, time, unicodedata
from pathlib import Path

import numpy as np
import soundfile as sf
import sherpa_onnx
from huggingface_hub import snapshot_download

CONFIG = {
    "model": "g-group-ai-lab/gipformer-68M-rnnt",
    "revision": "318322b009200f8e77a3751fa64638e5cc85a7dc",
    "int8": False,
    "decoding_method": "modified_beam_search",
    "num_threads": os.cpu_count() or 4,
    "batch_size": 32,
    "train_check_n": 150,  # số mẫu train (lượt cuối) để đo WER; 0 = bỏ qua
    "out_dir": "/kaggle/working",
}

def find_split(split):
    hits = sorted(glob.glob(f"/kaggle/input/**/{split}/{split}.jsonl", recursive=True))
    assert hits, f"không thấy {split}.jsonl trong /kaggle/input"
    return Path(hits[0]).parent

PUB = find_split("public_test")
TRAIN = find_split("train")
print(PUB, TRAIN, CONFIG["num_threads"], "threads")

In [ ]:
mdir = Path(snapshot_download(CONFIG["model"], revision=CONFIG["revision"],
                              allow_patterns=["*.onnx", "tokens.txt"]))
sfx = ".int8.onnx" if CONFIG["int8"] else ".onnx"
recognizer = sherpa_onnx.OfflineRecognizer.from_transducer(
    encoder=str(mdir / f"encoder{sfx}"),
    decoder=str(mdir / f"decoder{sfx}"),
    joiner=str(mdir / f"joiner{sfx}"),
    tokens=str(mdir / "tokens.txt"),
    num_threads=CONFIG["num_threads"],
    sample_rate=16000,
    feature_dim=80,
    decoding_method=CONFIG["decoding_method"],
)

def load_audio(path):
    x, sr = sf.read(path, dtype="float32", always_2d=True)
    x = x.mean(axis=1)
    if sr != 16000:
        import librosa
        x = librosa.resample(x, orig_sr=sr, target_sr=16000)
        sr = 16000
    return x, sr

def transcribe(paths):
    out = {}
    t0 = time.time()
    for i in range(0, len(paths), CONFIG["batch_size"]):
        chunk = paths[i:i + CONFIG["batch_size"]]
        streams, durs = [], []
        for p in chunk:
            x, sr = load_audio(p)
            s = recognizer.create_stream()
            s.accept_waveform(sr, x)
            streams.append(s)
            durs.append(len(x) / sr)
        recognizer.decode_streams(streams)
        for p, s, d in zip(chunk, streams, durs):
            out[p] = (s.result.text.strip(), d)
        done = i + len(chunk)
        if done % (CONFIG["batch_size"] * 10) == 0 or done == len(paths):
            print(f"{done}/{len(paths)} audio, {time.time() - t0:.0f}s")
    return out

In [ ]:
def norm(t):
    t = unicodedata.normalize("NFC", t).lower()
    t = re.sub(r"[^\w\s]", " ", t)
    return t.split()

def wer(ref, hyp):
    r, h = norm(ref), norm(hyp)
    d = list(range(len(h) + 1))
    for i in range(1, len(r) + 1):
        prev, d[0] = d[0], i
        for j in range(1, len(h) + 1):
            cur = min(d[j] + 1, d[j - 1] + 1, prev + (r[i - 1] != h[j - 1]))
            prev, d[j] = d[j], cur
    return d[len(h)], len(r)

# Kiểm nhanh trên train (có audio_text) trước khi chạy cả public_test
if CONFIG["train_check_n"]:
    items = [json.loads(l) for l in open(TRAIN / "train.jsonl", encoding="utf-8") if l.strip()]
    items = items[: CONFIG["train_check_n"]]
    res = transcribe([str(TRAIN / it["audio"]) for it in items])
    errs = toks = 0
    worst = []
    for it in items:
        hyp = res[str(TRAIN / it["audio"])][0]
        e, n = wer(it["audio_text"], hyp)
        errs += e; toks += n
        worst.append((e / max(n, 1), it["id"], it["audio_text"], hyp))
    worst.sort(reverse=True)
    report = {"n": len(items), "wer": round(errs / max(toks, 1), 4),
              "worst": [{"id": w[1], "ref": w[2], "hyp": w[3], "wer": round(w[0], 3)} for w in worst[:15]]}
    json.dump(report, open(f"{CONFIG['out_dir']}/asr_train_check.json", "w"), ensure_ascii=False, indent=1)
    print("WER train:", report["wer"])
    for w in report["worst"][:5]:
        print(w)

In [ ]:
items = [json.loads(l) for l in open(PUB / "public_test.jsonl", encoding="utf-8") if l.strip()]
rel = []
for it in items:
    rel.append(it["audio"])
    rel += [h["audio"] for h in it.get("history", []) if h.get("role") == "user" and h.get("audio")]
missing = [r for r in rel if not (PUB / r).is_file()]
print(len(items), "mẫu,", len(rel), "audio, thiếu", len(missing), missing[:5])

res = transcribe([str(PUB / r) for r in rel if (PUB / r).is_file()])
text = {r: res.get(str(PUB / r), ("", 0.0)) for r in rel}

with open(f"{CONFIG['out_dir']}/asr_public_test_files.jsonl", "w", encoding="utf-8") as f:
    for r in rel:
        f.write(json.dumps({"audio": r, "text": text[r][0], "dur_s": round(text[r][1], 2)}, ensure_ascii=False) + "\n")
with open(f"{CONFIG['out_dir']}/asr_public_test.jsonl", "w", encoding="utf-8") as f:
    for it in items:
        hist = [text[h["audio"]][0] for h in it.get("history", []) if h.get("role") == "user" and h.get("audio")]
        f.write(json.dumps({"id": it["id"], "audio_text": text[it["audio"]][0], "history_audio_text": hist},
                           ensure_ascii=False) + "\n")

empty = [r for r in rel if not text[r][0]]
print("transcript rỗng:", len(empty), empty[:10])
print(open(f"{CONFIG['out_dir']}/asr_public_test.jsonl", encoding="utf-8").read()[:1500])